# Part 2 — A transferable implicit transfer operator for proteins (TITO)

In part 1 you trained an implicit transfer operator (ITO) for **one** system, the 2D Wolfe–Quapp potential.
Here you use a **pre-trained, transferable** ITO, which was trained on many proteins and can be applied to proteins it has never seen.

**TITO / PLaTITO** ([paper](https://arxiv.org/abs/2602.11216), [project page](https://panosantoniadis.github.io/platito/), [code](https://github.com/PanosAntoniadis/platito)) learns the transition density

$$p(x_{t+\Delta t} \mid x_t, S, T, \Delta t)$$

* $x_t$: the protein conformation, **coarse-grained to the C$_\alpha$ atoms** (one bead per residue, coordinates in nm)
* $S$: the amino-acid sequence, $T$: the temperature, $\Delta t$: the lag time (in ns)

The model is a **conditional flow-matching** model, like your model from part 1. Given $x_t$, it turns Gaussian noise into a sample of $x_{t+\Delta t}$ by integrating an ODE.
It was trained on [mdCATH](https://www.nature.com/articles/s41597-024-04140-z), a large set of all-atom MD simulations of protein domains at 320, 348, 379, 413 and 450 K.
Applying the model over and over gives a trajectory with time step $\Delta t$ (Algorithm 3 in the paper).
$\Delta t$ can be nanoseconds, much longer than the femtosecond steps of MD.

We use **TITO-3M** (3 million parameters). It only uses the amino-acid identities, so it runs offline without any extra input.
The *PLaTITO* models add protein-language-model (ESM C) embeddings of the sequence, which improves transfer to new proteins.
Computing those embeddings needs an API token for the EvolutionaryScale Forge service. If you want to try this later, see the
[platito repository](https://github.com/PanosAntoniadis/platito) (optional extra, not needed today).

**Plan** (~1.5 h)
1. Run TITO-3M on a fast-folding protein and look at the trajectories.
2. Analyse them: RMSD, fraction of native contacts $Q$, radius of gyration, free-energy profiles, TICA.
3. Check the model as in part 1: stationarity (do different starting points end in the same ensemble?), Chapman–Kolmogorov self-consistency, and temperature dependence.

All computations run on your laptop's CPU. Runtimes are printed before the longer cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from minito import proteins as P

model = P.load_tito()  # TITO-3M, shipped in data/checkpoints/tito_3M.ckpt
print(f"TITO-3M loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

## 1. Choose a protein

The folder `data/proteins/` holds starting structures for 11 of the 12 fast-folding proteins of
[Lindorff-Larsen et al., *Science* 2011](https://www.science.org/doi/10.1126/science.1208351) (chignolin is not included).
For each protein there is a **folded** and an **unfolded** structure, and the temperature of the reference MD simulation.
The PLaTITO paper uses these proteins as out-of-distribution test systems.

In [ ]:
print(f"{'protein':12s} {'residues':>8s} {'T_MD (K)':>9s}")
for name, (n_res, temp) in P.list_proteins().items():
    print(f"{name:12s} {n_res:8d} {temp:9d}")

**Settings.** The defaults (villin headpiece, 16 trajectories × 50 steps of 10 ns) take about 1–3 minutes, depending on your laptop.
Runtime grows roughly linearly with `N_TRAJ`, `N_STEPS` and `ODE_STEPS`, and faster than linearly with the number of residues.
`LAG_NS` does not change the cost of a step, so a larger lag covers more simulated time for the same compute.

In [ ]:
PROTEIN = "villin"      # one of the proteins listed above
START = "unfolded"      # "unfolded" or "folded"
TEMPERATURE = P.reference_temperature(PROTEIN)  # in K; the model was trained on 320-450 K
LAG_NS = 10             # lag time Δt of one model step, in ns (integer; training used up to ~200 ns)
N_TRAJ = 16             # number of independent trajectories (run in parallel as one batch)
N_STEPS = 50            # number of model steps per trajectory -> N_STEPS * LAG_NS ns per trajectory
ODE_STEPS = 10          # ODE solver steps per flow-matching sample ...
ODE_METHOD = "midpoint" # ... midpoint (2 network calls per step); the paper uses 50 "euler" steps

start, folded = P.load_protein(PROTEIN, START)
print(f"{PROTEIN}: {start.n_residues} residues, sequence {P.sequence(start)}")
print(f"T = {TEMPERATURE} K, {N_TRAJ} trajectories x {N_STEPS * LAG_NS} ns")

<details><summary><b>Optional: use your own protein</b> (click to expand)</summary>

You can run any protein, e.g. one you work on. The model uses only the C$_\alpha$ atoms of the first chain.
Keep it small (≲ 60 residues), because cost grows quickly with length.
There is no unfolded structure, so start from the structure itself (`start = folded`).
To get an unfolded starting structure, you can run a few steps at high temperature (e.g. 450 K) first.

```python
path = "path/to/your.pdb"               # a local file, or (needs internet):
# path = P.fetch_structure("1L2Y")          # PDB ID (trp-cage NMR structure)
# path = P.fetch_structure("AF-P01542-F1")  # AlphaFold DB entry (crambin)
folded = P.load_ca(path)
start = folded
PROTEIN, TEMPERATURE = "my_protein", 320
print(folded.n_residues, P.sequence(folded))
```
</details>

In [ ]:
q0 = P.fraction_native_contacts(start.xyz, folded)[0]
r0 = P.rmsd(start.xyz, folded)[0]
print(f"start structure: RMSD to folded = {r0:.2f} nm, Q = {q0:.2f}")
P.plot_snapshots(np.stack([folded.xyz[0], start.xyz[0]]), [0, 1], reference=folded,
                 title="frame 0: folded reference, frame 1: start");

## 2. Generate trajectories

Each step calls the model once: it encodes $x_t$ (plus sequence, $T$, $\Delta t$) and integrates the flow-matching ODE from noise to $x_{t+\Delta t}$.
All `N_TRAJ` trajectories run together as one batch. The cell first times a single step, then runs the full simulation.

In [ ]:
sec = P.time_one_step(model, start, N_TRAJ, LAG_NS, TEMPERATURE, ODE_STEPS, ODE_METHOD)
print(f"one step: {sec:.1f} s  ->  estimated {sec * N_STEPS / 60:.1f} min for {N_STEPS} steps")

coords = P.simulate(model, start, N_TRAJ, N_STEPS, LAG_NS, TEMPERATURE, ODE_STEPS, ODE_METHOD)
print("coords:", coords.shape, "= (trajectories, frames, residues, xyz) in nm")

In [ ]:
out = P.save_trajectories(coords, start, f"{PROTEIN}_{START}_{TEMPERATURE}K", LAG_NS, reference=folded)
print("saved to", out)

The trajectories are saved as `topology.pdb` + `traj_XXX.dcd`. You can open them in ChimeraX, VMD or PyMOL, e.g. `vmd topology.pdb traj_000.dcd`.
Below is an animation of the first trajectory, aligned to the folded structure and coloured from N- (red) to C-terminus (blue).

*If the viewer stays empty, your browser could not load 3Dmol.js (it needs internet unless `data/3Dmol-min.js` exists).
The static snapshots in the next cell always work.*

In [ ]:
P.show_trajectory(coords[0], start, reference=folded)

In [ ]:
P.plot_snapshots(coords[0], [0, N_STEPS // 4, N_STEPS // 2, N_STEPS], reference=folded,
                 title=f"trajectory 0 (frames are {LAG_NS} ns apart)");

### Is the output physically sensible?

A coarse-grained model has no bonds to keep the chain together. It has to learn chain connectivity from data.
A simple check is the distance between consecutive C$_\alpha$ atoms, which is ≈ 0.38 nm in real proteins (≈ 0.29 nm for the rare *cis*-prolines).

In [ ]:
b = P.bond_lengths(coords[:, 1:])
plt.hist(b.ravel(), bins=100, range=(0.2, 0.5), density=True)
plt.axvline(0.38, color="k", ls="--")
plt.xlabel(r"C$_\alpha$–C$_\alpha$ distance (nm)"); plt.ylabel("density")
print(f"mean {b.mean():.3f} nm, std {b.std():.3f} nm")

> **Exercises**
> 1. *ODE solver.* Each model step integrates the flow-matching ODE from noise to a sample. Compare one step with `ode_steps=5, ode_method="euler"` against the default, e.g.
>    `x = P.propagate(model, start, coords[:, 0], LAG_NS, TEMPERATURE, ode_steps=5, ode_method="euler")` and then `P.bond_lengths(x)`.
>    Then run a whole trajectory (`P.simulate(...)`) with `ode_steps=20, ode_method="euler"`. It costs the same as the default (20 network calls per step), but the bond lengths slowly drift in long rollouts. Why do small errors per step matter so much when you apply the operator many times?
>    How does this relate to the discretisation error of an MD integrator?
> 2. *Unknown inputs.* The villin variant simulated by Lindorff-Larsen et al. contains two norleucines (NLE), which are not among the model's 20 amino acids.
>    We map them to methionine (M, a close isostere). Set `P.EXTRA_3_TO_1["NLE"] = "X"` (the "unknown residue" token), run one step from the unfolded villin structure, and look at the bond lengths around residues 23 and 28.
>    What does this tell you about using a transferable model on inputs that are rare or absent in its training data? (Set it back to `"M"` afterwards.)

## 3. Analysis

We look at three standard observables of folding:
* **RMSD** of the C$_\alpha$ atoms to the folded structure (after optimal superposition)
* **Fraction of native contacts $Q$** ([Best, Hummer & Eaton, *PNAS* 2013](https://www.pnas.org/doi/10.1073/pnas.1311599110)), here from C$_\alpha$ pairs at least 4 residues apart that are within 1 nm in the folded structure. $Q \approx 1$ when folded, small when unfolded.
* **Radius of gyration $R_g$**, which measures compactness

In [ ]:
rmsd = P.rmsd(coords, folded)
Q = P.fraction_native_contacts(coords, folded)
rg = P.radius_of_gyration(coords, folded)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, obs, label in zip(axes, [rmsd, Q, rg], ["RMSD (nm)", "Q", r"$R_g$ (nm)"]):
    P.plot_timeseries(obs, LAG_NS, ax=ax, ylabel=label)
    ax.plot(np.arange(obs.shape[1]) * LAG_NS, obs.mean(0), "k", lw=2, label="mean")
axes[1].legend(); plt.tight_layout()

**Free-energy profiles.** If the trajectories have reached equilibrium, the histogram of an observable gives the free energy $F(x) = -k_BT \ln p(x)$.
We pool the second half of all trajectories and report $F$ in units of $k_BT$.

In [ ]:
half = N_STEPS // 2
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x, F = P.free_energy_1d(Q[:, half:], bins=25, range=(0, 1))
axes[0].plot(x, F, "o-"); axes[0].set_xlabel("Q"); axes[0].set_ylabel(r"F / $k_BT$")
P.plot_free_energy_2d(Q[:, half:], rg[:, half:], bins=15, ax=axes[1])
axes[1].set_xlabel("Q"); axes[1].set_ylabel(r"$R_g$ (nm)")
plt.tight_layout()

**TICA.** Time-lagged independent component analysis finds the linear combinations of features that decorrelate most slowly in time.
These are the standard coordinates for comparing free-energy surfaces, and the paper uses them to benchmark TITO against MD.
As features we use C$_\alpha$–C$_\alpha$ distances between 16 evenly spaced residues (using all pairs would overfit our few hundred frames), with `deeptime`.
Here we fit TICA on the generated trajectories themselves, because the reference MD of these proteins cannot be redistributed.

In [ ]:
tica = P.fit_tica([coords], lagtime=1)  # lagtime in frames, i.e. LAG_NS ns
proj = P.tica_project(tica, coords)      # (trajectories, frames, 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
P.plot_free_energy_2d(proj[:, half:, 0], proj[:, half:, 1], bins=20, ax=axes[0])
axes[0].set_xlabel("TIC 1"); axes[0].set_ylabel("TIC 2"); axes[0].set_title("free energy (2nd half)")
sc = axes[1].scatter(proj[..., 0].ravel(), proj[..., 1].ravel(), c=Q.ravel(), s=4, cmap="coolwarm_r", vmin=0, vmax=1)
plt.colorbar(sc, ax=axes[1], label="Q"); axes[1].set_xlabel("TIC 1"); axes[1].set_ylabel("TIC 2")
axes[1].set_title("all frames, coloured by Q")
plt.tight_layout()

> **Questions**
> 1. Does villin fold within the simulated time? Compare with the folding time of ≈ 3 μs found by MD at 360 K (Lindorff-Larsen et al.). How many folding events would you expect to see in $16 \times 0.5\,\mu s$?
> 2. Which TIC separates folded from unfolded structures? Is TIC 1 always the folding coordinate?
> 3. Why can we generate 500 ns of trajectory in a minute, when all-atom MD of villin would take hours or days for the same time? What do we lose by coarse-graining and taking large time steps?

## 4. Evaluating the model

In part 1 you compared the ITO to a reference simulation. For these proteins we have no reference MD here, but we can still test the
properties that any correct transfer operator must have.

### 4a. Stationarity: do different starting points reach the same ensemble?

If the model has a unique stationary (Boltzmann-like) distribution and our trajectories are long enough, then trajectories started from the **folded** and from the **unfolded** structure
must end up with the same distribution of $Q$.

In [ ]:
other = "folded" if START == "unfolded" else "unfolded"
start_other, _ = P.load_protein(PROTEIN, other)
coords_other = P.simulate(model, start_other, N_TRAJ, N_STEPS, LAG_NS, TEMPERATURE, ODE_STEPS, ODE_METHOD, seed=1)
Q_other = P.fraction_native_contacts(coords_other, folded)

t = np.arange(N_STEPS + 1) * LAG_NS
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for q, lab in [(Q, START), (Q_other, other)]:
    axes[0].plot(t, q.mean(0), lw=2, label=f"start: {lab}")
    axes[0].fill_between(t, np.percentile(q, 25, 0), np.percentile(q, 75, 0), alpha=0.2)
    x, F = P.free_energy_1d(q[:, half:], bins=25, range=(0, 1))
    axes[1].plot(x, F, "o-", label=f"start: {lab} (2nd half)")
axes[0].set_xlabel("time (ns)"); axes[0].set_ylabel("Q (mean, 25-75%)"); axes[0].legend()
axes[1].set_xlabel("Q"); axes[1].set_ylabel(r"F / $k_BT$"); axes[1].legend()
plt.tight_layout()

> **Questions**
> 1. Have the two ensembles converged? If not, which one relaxes more slowly, and what does that tell you about the slowest time scale of the model?
> 2. The model's stationary distribution need not be the true Boltzmann distribution. How could you test this if you had reference MD (think of part 1)?
> 3. **Exercise:** Increase `LAG_NS` (e.g. 50 or 100 ns) and keep `N_STEPS` fixed. Do the ensembles converge faster in wall-clock time? Does the stationary distribution change?

### 4b. Chapman–Kolmogorov self-consistency

A Markovian transfer operator must satisfy $\mathcal{T}(2\tau) = \mathcal{T}(\tau)\,\mathcal{T}(\tau)$.
TITO is conditioned on $\Delta t$, so we can check this directly. Starting from the **same** structures, we compare
* $k$ steps with lag $\tau$, and
* one step with lag $k\tau$.

If the model were exactly Markovian and self-consistent, both would give the same distribution of $x_{t+k\tau}$. We compare the distributions of $Q$ and $R_g$.

In [ ]:
TAU = LAG_NS
K = 4                          # compare 4 steps of TAU with 1 step of 4*TAU
N_CK = 64                      # samples per distribution
x0 = np.repeat(start.xyz[:1], N_CK, axis=0)

torch.manual_seed(0)
x_long = P.propagate(model, start, x0, K * TAU, TEMPERATURE, ODE_STEPS, ODE_METHOD)   # one step of K*TAU
x_short = x0
for _ in range(K):                                                       # K steps of TAU
    x_short = P.propagate(model, start, x_short, TAU, TEMPERATURE, ODE_STEPS, ODE_METHOD)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, f, lab in [(axes[0], lambda x: P.fraction_native_contacts(x, folded), "Q"),
                   (axes[1], lambda x: P.radius_of_gyration(x, folded), r"$R_g$ (nm)")]:
    a, b = f(x_long), f(x_short)
    bins = np.linspace(min(a.min(), b.min()), max(a.max(), b.max()), 20)
    ax.hist(a, bins=bins, alpha=0.5, density=True, label=f"1 step of {K * TAU} ns")
    ax.hist(b, bins=bins, alpha=0.5, density=True, label=f"{K} steps of {TAU} ns")
    ax.set_xlabel(lab); ax.legend()
plt.tight_layout()

> **Questions**
> 1. Do the two distributions agree? With only 64 samples, how large a difference would you expect from noise alone (try another seed)?
> 2. When would you expect Chapman–Kolmogorov violations for a model of MD? Think of (i) a lag that is too short for the coarse-grained dynamics to be Markovian, and (ii) a model that is inaccurate at small lags, where errors pile up over many steps.
> 3. In part 1 you did the CK test with a Markov state model. How would you set up the same test here, e.g. using states defined by $Q$?

### 4c. Temperature dependence

TITO is conditioned on temperature. Proteins should unfold as the temperature goes up.
Below, we start from the **folded** structure and measure the fraction of folded frames (here: $Q > 0.6$) after a short simulation at several temperatures.
This is a cheap, rough estimate. The fraction is not yet at equilibrium.

In [ ]:
TEMPS = [320, 350, 380, 450]     # within the training range 320-450 K
folded_start, _ = P.load_protein(PROTEIN, "folded")
frac_folded = []
for temp in TEMPS:
    c = P.simulate(model, folded_start, 8, 10, 50, temp, ODE_STEPS, ODE_METHOD, progress=False)  # 8 traj x 10 steps x 50 ns
    frac_folded.append(np.mean(P.fraction_native_contacts(c[:, 5:], folded) > 0.6))
    print(f"T = {temp} K: fraction folded = {frac_folded[-1]:.2f}")
plt.plot(TEMPS, frac_folded, "o-"); plt.xlabel("T (K)"); plt.ylabel("fraction with Q > 0.6"); plt.ylim(-0.05, 1.05);

> **Questions & exercises** (pick what interests you)
> 1. Where does the model put the melting temperature? The model never saw this protein, and it was trained on mdCATH domains, not on fast folders.
>    The reference MD temperatures (`T_MD` in the table above) were chosen close to each protein's melting temperature, so both folded and unfolded states are populated in MD. Is that also true for TITO-3M at `T_MD`?
> 2. Try a temperature outside the training range (e.g. 280 K or 500 K). What happens? Why is extrapolating in a *condition* risky?
> 3. Try a different protein, e.g. `trp_cage` (20 residues, very fast), `bba`, `wwdomain` (a β-sheet protein), or `lambda` (80 residues, slow). Which ones does TITO-3M fold? Is that related to secondary-structure type or size?
> 4. Run your own protein (see the optional cell in section 1). Does the folded state stay stable at 320 K?
> 5. Which effects does a C$_\alpha$-only model leave out (side-chain packing, solvent, ...)? How could you check whether the kinetics TITO predicts are correct, not only the free energies?
> 6. In the paper, PLaTITO with language-model embeddings transfers better than TITO. Why might evolutionary information from a protein language model help an MD surrogate?